In [1]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 1 — BOOTSTRAP
# ═══════════════════════════════════════════════════════════════════════

import importlib, json, os, sys, time
from pathlib import Path

import numpy as np
import pandas as pd

# Flip to True if you need verbose diagnostics.
DEBUG = False

def _log(msg=""):
    """Always printed — essential developer output."""
    print(msg)

def _dev(msg=""):
    """Printed only when DEBUG=True."""
    if DEBUG:
        print(msg)

def _section(title):
    _log()
    _log("-" * 72)
    _log(title)
    _log("-" * 72)

# ── Pipeline modules ───────────────────────────────────────────────────
import _shared
importlib.reload(_shared)
import Extraction as EX
importlib.reload(EX)
import Probe as probe
importlib.reload(probe)

from _shared import (
    AMIRALI_MOUNT, DATASETS_ROOT,
    HIDDEN_STATES_ROOT, INTEREX_ROOT, PROBE_ROOT,
    model_slug,
)

_section("STEP 1/6 — Paths and system")

for label, p in (
    ("Mount",           AMIRALI_MOUNT),
    ("Datasets root",   DATASETS_ROOT),
    ("Hidden states",   HIDDEN_STATES_ROOT),
    ("InterEx (probe)", INTEREX_ROOT),
):
    if not p.is_dir():
        raise RuntimeError(f"Required path missing: {label} = {p}")
    _log(f"  [OK] {label:16s} {p}")

import platform, torch
_log(f"  Platform       : {platform.platform()}")
_log(f"  Python         : {platform.python_version()}")
_log(f"  PyTorch        : {torch.__version__}")
_log(f"  CPU threads    : {torch.get_num_threads()}")
try:
    import psutil
    vm = psutil.virtual_memory()
    _log(f"  RAM total      : {vm.total / 1e9:.2f} GB")
    _log(f"  RAM available  : {vm.available / 1e9:.2f} GB")
except ImportError:
    _log("  RAM info       : psutil not installed")
_log(f"  Probe device   : {probe.choose_device()}")


------------------------------------------------------------------------
STEP 1/6 — Paths and system
------------------------------------------------------------------------
  [OK] Mount            /Volumes/Amirali
  [OK] Datasets root    /Volumes/Amirali/datasets
  [OK] Hidden states    /Volumes/Amirali/hidden_states
  [OK] InterEx (probe)  /Volumes/Amirali/interEx
  Platform       : macOS-14.6.1-x86_64-i386-64bit
  Python         : 3.12.0
  PyTorch        : 2.2.2
  CPU threads    : 4
  RAM total      : 8.59 GB
  RAM available  : 3.47 GB
  Probe device   : cpu


In [2]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 2 — DATASET DISCOVERY
# ═══════════════════════════════════════════════════════════════════════

_section("STEP 2/6 — Processed dataset discovery")

DATASETS, CSV_HASHES = EX.discover_processed_datasets(show_info=False)
assert DATASETS, "No processed datasets discovered."

EXCLUDE_DATASETS = {"amazon_polarity"}

DATASETS   = {k: v for k, v in DATASETS.items()   if k not in EXCLUDE_DATASETS}
CSV_HASHES = {k: v for k, v in CSV_HASHES.items() if k not in EXCLUDE_DATASETS}

# Contract checks
for name, df in DATASETS.items():
    assert list(df.columns) == list(EX.PROCESSED_COLUMNS), \
        f"{name}: bad columns {list(df.columns)}"
    assert df.index.is_unique and df.index[0] == 0 and df.index[-1] == len(df) - 1, \
        f"{name}: index is not a clean RangeIndex"
    assert df["label"].map(lambda x: isinstance(x, list) and len(x) > 0).all(), \
        f"{name}: labels must be non-empty lists"
    assert df["clean_text"].notna().all(), f"{name}: null clean_text"

for name, df in DATASETS.items():
    _log(f"  {name:22s} {len(df):>8,} rows  sha256={CSV_HASHES[name][:12]}")
_log(f"  -> {len(DATASETS)} dataset(s) satisfy the contract.")


------------------------------------------------------------------------
STEP 2/6 — Processed dataset discovery
------------------------------------------------------------------------
  emotion                  19,999 rows  sha256=5306dde7d824
  goemo                    54,039 rows  sha256=567885691ac5
  isear                     7,532 rows  sha256=075d2d987106
  sst2                     67,855 rows  sha256=fadcbf89bc37
  tweet_eval_emotion        5,027 rows  sha256=2791d0b2c71d
  -> 5 dataset(s) satisfy the contract.


In [3]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 3 — CONTRACTS AND FAST PROBE SPECS
# ═══════════════════════════════════════════════════════════════════════

_section("STEP 3/6 — Contracts and probes")

DATASET_CONTRACTS = {
    name: probe.DatasetContract(**_shared.contract_dict_for(name))
    for name in DATASETS
}

for name, c in DATASET_CONTRACTS.items():
    n_cls = len(c.class_order) if c.class_order else "derived"
    _log(f"  {name:22s} target={c.target_type:12s} "
         f"task={c.task_type:12s} classes={n_cls}")

# ── Fast probe specs ───────────────────────────────────────────────────
# vs. the default configuration:
#   logistic.max_iter    3000 -> 1000
#   mlp.epochs             80 -> 40
#   mlp.patience           12 -> 6
#   mlp.batch_size        256 -> 512
# These do not change the shape of the layer curve; they only reduce
# the wall time of each fit.
probes = [
    probe.ProbeSpec(
        name="linear_logistic",
        type="logistic", complexity="linear", standardize=True,
        C=1.0, max_iter=1000,
        selection_metric="macro_f1",
    ),
    probe.ProbeSpec(
        name="mlp_1_hidden",
        type="mlp", complexity="1_hidden", standardize=True,
        hidden_dims=["0.5d"],
        learning_rate=1e-3, weight_decay=1e-4,
        epochs=40, batch_size=512, patience=6,
        selection_metric="macro_f1",
    ),
    probe.ProbeSpec(
        name="mlp_2_hidden",
        type="mlp", complexity="2_hidden", standardize=True,
        hidden_dims=["0.5d", "0.25d"],
        learning_rate=1e-3, weight_decay=1e-4,
        epochs=40, batch_size=512, patience=6,
        selection_metric="macro_f1",
    ),
    probe.ProbeSpec(
        name="mlp_3_hidden",
        type="mlp", complexity="3_hidden", standardize=True,
        hidden_dims=["0.5d", "0.25d", "0.125d"],
        learning_rate=1e-3, weight_decay=1e-4,
        epochs=40, batch_size=512, patience=6,
        selection_metric="macro_f1",
    ),
]

_log(f"  -> {len(DATASET_CONTRACTS)} contract(s), {len(probes)} probe(s).")
for p in probes:
    _log(f"     {p.name:20s} type={p.type:8s} complexity={p.complexity}")


------------------------------------------------------------------------
STEP 3/6 — Contracts and probes
------------------------------------------------------------------------
  emotion                target=custom       task=single_label classes=6
  goemo                  target=goemotions   task=multi_label  classes=28
  isear                  target=isear        task=single_label classes=7
  sst2                   target=custom       task=single_label classes=2
  tweet_eval_emotion     target=custom       task=single_label classes=4
  -> 5 contract(s), 4 probe(s).
     linear_logistic      type=logistic complexity=linear
     mlp_1_hidden         type=mlp      complexity=1_hidden
     mlp_2_hidden         type=mlp      complexity=2_hidden
     mlp_3_hidden         type=mlp      complexity=3_hidden


In [4]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 4 — DISCOVERY HELPERS
# ═══════════════════════════════════════════════════════════════════════

def discover_extraction_pairs(hidden_states_root: Path = HIDDEN_STATES_ROOT) -> pd.DataFrame:
    """Every (model, dataset) with a completed extraction under hidden_states/."""
    rows = []
    if not hidden_states_root.is_dir():
        return pd.DataFrame(columns=["model", "dataset", "artifact_dir"])
    for model_dir in sorted(hidden_states_root.iterdir()):
        if not model_dir.is_dir() or model_dir.name.startswith("_"):
            continue
        for dataset_dir in sorted(model_dir.iterdir()):
            if not dataset_dir.is_dir():
                continue
            meta_path = dataset_dir / "extraction.json"
            if not (dataset_dir / "hidden_states.npy").is_file() or not meta_path.is_file():
                continue
            try:
                meta = json.loads(meta_path.read_text())
            except Exception:
                continue
            rows.append({
                "model":        meta.get("model", {}).get("name") or model_dir.name,
                "dataset":      meta.get("dataset", {}).get("name") or dataset_dir.name,
                "artifact_dir": str(dataset_dir),
            })
    return pd.DataFrame(rows)


def discover_probe_runs(interex_root: Path = INTEREX_ROOT) -> pd.DataFrame:
    """Every probe run registered under interEx/<slug>/<dataset>/index.json."""
    cols = ["model", "dataset", "run_key", "trial_hash",
            "probes", "results_csv", "task_type", "n_classes"]
    rows = []
    if not interex_root.is_dir():
        return pd.DataFrame(columns=cols)
    for model_dir in sorted(interex_root.iterdir()):
        if not model_dir.is_dir() or model_dir.name.startswith("_"):
            continue
        for dataset_dir in sorted(model_dir.iterdir()):
            if not dataset_dir.is_dir():
                continue
            index = dataset_dir / "index.json"
            if not index.is_file():
                continue
            try:
                payload = json.loads(index.read_text())
            except Exception:
                continue
            for entry in payload.get("runs", []):
                rows.append({
                    "model":       entry["model"],
                    "dataset":     entry["dataset"],
                    "run_key":     entry["run_key"],
                    "trial_hash":  entry["trial_hash"],
                    "probes":      "+".join(entry["probes"]),
                    "results_csv": str(dataset_dir / entry["results_csv"]),
                    "task_type":   entry["task_type"],
                    "n_classes":   entry["n_classes"],
                })
    return pd.DataFrame(rows, columns=cols)


_section("STEP 4/6 — Discovery helpers ready")


------------------------------------------------------------------------
STEP 4/6 — Discovery helpers ready
------------------------------------------------------------------------


In [5]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 5 — PRE-FLIGHT
# ═══════════════════════════════════════════════════════════════════════

_section("STEP 5/6 — Pre-flight")

# 1. Any extraction artifacts at all?
pairs = discover_extraction_pairs()
if pairs.empty:
    raise RuntimeError(
        "No extraction artifacts found under hidden_states/. "
        "Run __Extraction_Runner.ipynb first."
    )
_log(f"  OK  {len(pairs)} extraction artifact(s) on disk")

# 2. Every artifact has a contract?
known_pairs = pairs[pairs["dataset"].isin(DATASET_CONTRACTS.keys())].copy()
skipped     = pairs[~pairs["dataset"].isin(DATASET_CONTRACTS.keys())]
if not skipped.empty:
    _log(f"  !   {len(skipped)} artifact(s) skipped (no contract):")
    for _, r in skipped.head(8).iterrows():
        _log(f"        {r['model']:<45} {r['dataset']}")
if known_pairs.empty:
    raise RuntimeError("No (model, dataset) pairs resolved to a known contract.")

# 3. Contract task_type matches extraction metadata?
_mismatches = []
for _, r in known_pairs.iterrows():
    try:
        meta = json.loads((Path(r["artifact_dir"]) / "extraction.json").read_text())
    except Exception:
        continue
    ext_task = meta.get("extraction", {}).get("task_type")
    ct = DATASET_CONTRACTS[r["dataset"]].task_type
    if ext_task and ext_task not in (ct, "auto", "unknown"):
        _mismatches.append((r["model"], r["dataset"], ext_task, ct))
if _mismatches:
    for m, d, et, ct in _mismatches:
        _log(f"  X  {m}/{d}: extraction={et} contract={ct}")
    raise RuntimeError(f"{len(_mismatches)} task_type mismatch(es)")
_log("  OK  contract task_type matches every extraction")

# 4. Every probe spec valid for every task type present?
_task_types = {DATASET_CONTRACTS[d].task_type for d in known_pairs["dataset"].unique()}
for spec in probes:
    for tt in _task_types:
        probe.validate_probe_spec(spec, tt)
_log(f"  OK  all probes valid for task types {sorted(_task_types)}")

# ── Target table ──────────────────────────────────────────────────────
_section("STEP 5/6 — Target list")

target_rows = []
for _, r in known_pairs.iterrows():
    c = DATASET_CONTRACTS[r["dataset"]]
    target_rows.append({
        "model":     r["model"].split("/")[-1],
        "dataset":   r["dataset"],
        "task_type": c.task_type,
        "classes":   len(c.class_order) if c.class_order else "derived",
    })
target_df = pd.DataFrame(target_rows).sort_values(["dataset", "model"]).reset_index(drop=True)
_log(target_df.to_string(index=False))

# ── Job count and time estimate ──────────────────────────────────────
N_PAIRS     = len(known_pairs)
AVG_LAYERS  = 25
OLD_FITS    = 4 * AVG_LAYERS * len(probes) * (1 + 3)   # old config
NEW_FITS    = 2 * AVG_LAYERS * len(probes) * (1 + 1)   # new config

_log()
_log(f"  Pairs                : {N_PAIRS}")
_log(f"  Fits per pair        : {OLD_FITS} -> {NEW_FITS}  "
     f"({OLD_FITS / NEW_FITS:.1f}x fewer)")
_log(f"  Total fits           : {OLD_FITS * N_PAIRS:,} -> {NEW_FITS * N_PAIRS:,}")
_log(f"  Per-fit speedup      : ~3-8x  (5000 rows vs full dataset)")
_log(f"  Expected wall time   : many days -> ~1.5-2 days")
_log()


------------------------------------------------------------------------
STEP 5/6 — Pre-flight
------------------------------------------------------------------------
  OK  20 extraction artifact(s) on disk
  OK  contract task_type matches every extraction
  OK  all probes valid for task types ['multi_label', 'single_label']

------------------------------------------------------------------------
STEP 5/6 — Target list
------------------------------------------------------------------------
            model            dataset    task_type  classes
       Qwen2-0.5B            emotion single_label        6
     Qwen2.5-0.5B            emotion single_label        6
  Qwen3-0.6B-Base            emotion single_label        6
bert-base-uncased            emotion single_label        6
       Qwen2-0.5B              goemo  multi_label       28
     Qwen2.5-0.5B              goemo  multi_label       28
  Qwen3-0.6B-Base              goemo  multi_label       28
bert-base-uncased            

In [6]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 6 — LAUNCH FAST PROBE MATRIX
# ═══════════════════════════════════════════════════════════════════════

REPEATS        = 2
MAX_SAMPLES    = 5000
CONTROL_REPS   = 1
VERBOSE_PY     = 0                # 0 silent | 1 sections | 3 per-fit
EXPERIMENT_ID  = "probe_fast_v1"
CHECKPOINT_DIR = PROBE_ROOT / "_matrix_checkpoint_fast"

_section("STEP 6/6 — Launch")
_log(f"  experiment_id : {EXPERIMENT_ID}")
_log(f"  pairs         : {len(known_pairs)}")
_log(f"  repeats       : {REPEATS}")
_log(f"  max_samples   : {MAX_SAMPLES}")
_log(f"  control_reps  : {CONTROL_REPS}")
_log(f"  checkpoint    : {CHECKPOINT_DIR}")

entries = []
for _, r in known_pairs.iterrows():
    entries.append({
        "model":        r["model"],
        "dataset":      r["dataset"],
        "artifact_dir": r["artifact_dir"],
        "contract":     DATASET_CONTRACTS[r["dataset"]],
        "dataset_df":   DATASETS[r["dataset"]],
    })

_t0 = time.perf_counter()

full_results = probe.run_matrix(
    entries,
    experiment_id=EXPERIMENT_ID,
    probes=probes,
    repeats=REPEATS,
    max_samples=MAX_SAMPLES,
    verbose=VERBOSE_PY,
    checkpoint_dir=CHECKPOINT_DIR,
    shuffled_label_control=True,
    shuffled_control_repeats=CONTROL_REPS,
)

_elapsed = time.perf_counter() - _t0
_section("STEP 6/6 — Matrix finished")
_log(f"  elapsed : {_elapsed/60:.1f} min")
_log(f"  rows    : {len(full_results):,}" if full_results is not None else "  rows    : 0")


------------------------------------------------------------------------
STEP 6/6 — Launch
------------------------------------------------------------------------
  experiment_id : probe_fast_v1
  pairs         : 20
  repeats       : 2
  max_samples   : 5000
  control_reps  : 1
  checkpoint    : /Volumes/Amirali/interEx/_matrix_checkpoint_fast
[validate] Checkpoint is consistent.
[matrix] 1/20 | Qwen/Qwen2-0.5B | emotion | 7ab14fee
[checkpoint] Resuming from 7ab14fee2ef1_layer_probe_results.csv
[matrix] 2/20 | Qwen/Qwen2-0.5B | goemo | 5302b46b
[checkpoint] Resuming from 5302b46bc7f1_layer_probe_results.csv
[matrix] 3/20 | Qwen/Qwen2-0.5B | isear | e2dc842f
[checkpoint] Resuming from e2dc842f4eb0_layer_probe_results.csv
[matrix] 4/20 | Qwen/Qwen2-0.5B | tweet_eval_emotion | 647f98f9
[checkpoint] Resuming from 647f98f9c3ea_layer_probe_results.csv
[matrix] 5/20 | Qwen/Qwen2-1.5B | isear | a9e6ccef
[checkpoint] Resuming from a9e6ccef37ef_layer_probe_results.csv
[matrix] 6/20 | Qwen/Qwen

Probing:   0%|          | 0/464 [00:00<?, ?fit/s]

[probe +    2.87s] ================================================================================================
[probe +    2.87s] PROBING EXPERIMENT
[probe +    2.87s] ================================================================================================
[probe +    2.87s] Question: how recoverable is the target from each frozen hidden-state layer?
[probe +    2.87s] repeats=2 | max_samples=5000 | layers=29 | probes=4


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 7 — POST-RUN SUMMARY
# ═══════════════════════════════════════════════════════════════════════

_section("POST-RUN — Best layer per probe")

df = full_results if 'full_results' in globals() and full_results is not None \
     and not full_results.empty else pd.DataFrame()

if df.empty:
    runs = discover_probe_runs()
    if not runs.empty:
        frames = []
        for _, run in runs.iterrows():
            p = Path(run["results_csv"])
            if not p.is_file():
                continue
            rdf = pd.read_csv(p)
            if "model"   not in rdf.columns: rdf["model"]   = run["model"]
            if "dataset" not in rdf.columns: rdf["dataset"] = run["dataset"]
            frames.append(rdf)
        df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

if df.empty:
    _log("  No probe results on disk yet.")
else:
    best_per_probe = (
        df.loc[df.groupby(["probe", "model", "dataset"])["test_macro_f1"].idxmax()]
          .sort_values(["dataset", "model", "probe"])
    )
    _log(best_per_probe[[
        "probe", "model", "dataset", "layer_index",
        "test_macro_f1", "probe_score",
    ]].to_string(index=False))

    _section("POST-RUN — Best Macro-F1 matrix")
    pivot = best_per_probe.pivot_table(
        index=["model", "dataset"], columns="probe", values="test_macro_f1",
    )
    _log(pivot.to_string(float_format=lambda x: f"{x:.4f}"))
    _log(f"  rows: {len(df):,}")

------------------------------------------------------------
STEP 2/6 — Processed dataset discovery
------------------------------------------------------------------------
  emotion                  19,999 rows  sha256=5306dde7d824
  goemo                    54,039 rows  sha256=567885691ac5
  isear                     7,532 rows  sha256=075d2d987106
  sst2                     67,855 rows  sha256=fadcbf89bc37
  tweet_eval_emotion        5,027 rows  sha256=2791d0b2c71d
  -> 5 dataset(s) satisfy the contract.

------------------------------------------------------------------------
STEP 3/6 — Contracts and probes
------------------------------------------------------------------------
  emotion                target=custom       task=single_label classes=6
  goemo                  target=goemotions   task=multi_label  classes=28
  isear                  target=isear        task=single_label classes=7
  sst2                   target=custom       task=single_label classes=2
  tweet_eval_emotion     target=custom       task=single_label classes=4
  -> 5 contract(s), 4 probe(s).
     linear_logistic      type=logistic complexity=linear
     mlp_1_hidden         type=mlp      complexity=1_hidden
     mlp_2_hidden         type=mlp      complexity=2_hidden
     mlp_3_hidden         type=mlp      complexity=3_hidden

------------------------------------------------------------------------
STEP 5/6 — Pre-flight
------------------------------------------------------------------------
  OK  18 extraction artifact(s) on disk
  OK  contract task_type matches every extraction
  OK  all probes valid for task types ['multi_label', 'single_label']

------------------------------------------------------------------------
STEP 5/6 — Target list
------------------------------------------------------------------------
            model            dataset    task_type  classes
       Qwen2-0.5B            emotion single_label        6
     Qwen2.5-0.5B            emotion single_label        6
  Qwen3-0.6B-Base            emotion single_label        6
bert-base-uncased            emotion single_label        6
       Qwen2-0.5B              goemo  multi_label       28
     Qwen2.5-0.5B              goemo  multi_label       28
  Qwen3-0.6B-Base              goemo  multi_label       28
bert-base-uncased              goemo  multi_label       28
       Qwen2-0.5B              isear single_label        7
     Qwen2.5-0.5B              isear single_label        7
  Qwen3-0.6B-Base              isear single_label        7
bert-base-uncased              isear single_label        7
     Qwen2.5-0.5B               sst2 single_label        2
...
  Total fits           : 28,800 -> 7,200
  Per-fit speedup      : ~3-8x  (5000 rows vs full dataset)
  Expected wall time   : many days -> ~1.5-2 days

Output is truncated. View as a scrollable element or open in a text editor. Adjust cell output settings...

------------------------------------------------------------------------
STEP 6/6 — Launch
------------------------------------------------------------------------
  experiment_id : probe_fast_v1
  pairs         : 18
  repeats       : 2
  max_samples   : 5000
  control_reps  : 1
  checkpoint    : /Volumes/Amirali/interEx/_matrix_checkpoint_fast
[validate] Checkpoint file not found.
[matrix] 1/18 | Qwen/Qwen2-0.5B | emotion | 7ab14fee
[probe +    0.00s] ================================================================================================
[probe +    0.00s] INITIALISING UNIFIED HIDDEN-STATE PROBE
[probe +    0.00s] ================================================================================================
[probe +    0.22s] Starting new trial: /Volumes/Amirali/interEx/Qwen2-0.5B/emotion/Qwen-Qwen2-0.5B__emotion__linear_logistic+mlp_1_hidden+mlp_2_hidden+mlp_3_hidden__L25__R2__S5000__h7ab14fee2e
[probe +    3.35s] Model: Qwen/Qwen2-0.5B
[probe +    3.35s] Dataset artifact: emotion
[probe +    3.35s] Hidden-state shape: (19999, 25, 896)
[probe +    3.35s] Task type: single_label | classes: 6
[probe +    3.35s] Selected layers: 25 | device: cpu
[probe +    3.35s] Alignment: text=verified | labels=unverified
[probe +    3.45s] Resumed with 8 completed jobs.
Probing: 100%
 400/400 [1:04:37<00:00,  8.79s/fit]
[probe +    3.91s] ================================================================================================
[probe +    3.91s] PROBING EXPERIMENT
[probe +    3.91s] ================================================================================================
[probe +    3.91s] Question: how recoverable is the target from each frozen hidden-state layer?
[probe +    3.91s] repeats=2 | max_samples=5000 | layers=25 | probes=4
[probe + 3881.43s] All jobs completed. Generating final outputs.
[probe + 3901.96s] Complete run metadata saved: /Volumes/Amirali/interEx/Qwen2-0.5B/emotion/Qwen-Qwen2-0.5B__emotion__linear_logistic+mlp_1_hidden+mlp_2_hidden+mlp_3_hidden__L25__R2__S5000__h7ab14fee2e/complete_run_metadata.json
[probe + 3901.97s] ================================================================================================
[probe + 3901.97s] FINAL RESULT
[probe + 3901.97s] ================================================================================================
[probe + 3901.97s] Final best layer table:
          probe  layer_index  probe_score_mean  test_macro_f1_mean
linear_logistic            0          0.595476            0.609972
   mlp_1_hidden            1          0.635676            0.648797
   mlp_2_hidden            1          0.617494            0.648968
   mlp_3_hidden            1          0.593631            0.620774
[probe + 3902.06s] Output directory: /Volumes/Amirali/interEx/Qwen2-0.5B/emotion/Qwen-Qwen2-0.5B__emotion__linear_logistic+mlp_1_hidden+mlp_2_hidden+mlp_3_hidden__L25__R2__S5000__h7ab14fee2e
[matrix] 2/18 | Qwen/Qwen2-0.5B | goemo | 5302b46b
[probe +    0.00s] ================================================================================================
[probe +    0.00s] INITIALISING UNIFIED HIDDEN-STATE PROBE
[probe +    0.00s] ================================================================================================
[probe +    5.61s] Starting new trial: /Volumes/Amirali/interEx/Qwen2-0.5B/goemo/Qwen-Qwen2-0.5B__goemo__linear_logistic+mlp_1_hidden+mlp_2_hidden+mlp_3_hidden__L25__R2__S5000__h5302b46bc7
[probe +    6.24s] Model: Qwen/Qwen2-0.5B
[probe +    6.24s] Dataset artifact: goemo
[probe +    6.24s] Hidden-state shape: (54039, 25, 896)
...
[probe +    6.24s] Selected layers: 25 | device: cpu
[probe +    6.24s] Alignment: text=verified | labels=unverified
[probe +    7.77s] No valid progress file found, starting fresh.
[probe +    7.77s] Resumed with 0 completed jobs.
Output is truncated. View as a scrollable element or open in a text editor. Adjust cell output settings...
Probing: 100%
 400/400 [2:29:18<00:00, 15.01s/fit]
[probe +    7.93s] ================================================================================================
[probe +    7.93s] PROBING EXPERIMENT
[probe +    7.93s] ================================================================================================
[probe +    7.93s] Question: how recoverable is the target from each frozen hidden-state layer?
[probe +    7.93s] repeats=2 | max_samples=5000 | layers=25 | probes=4
[probe + 8966.02s] All jobs completed. Generating final outputs.
[probe + 8981.93s] Complete run metadata saved: /Volumes/Amirali/interEx/Qwen2-0.5B/goemo/Qwen-Qwen2-0.5B__goemo__linear_logistic+mlp_1_hidden+mlp_2_hidden+mlp_3_hidden__L25__R2__S5000__h5302b46bc7/complete_run_metadata.json
[probe + 8982.00s] ================================================================================================
[probe + 8982.00s] FINAL RESULT
[probe + 8982.00s] ================================================================================================
[probe + 8982.00s] Final best layer table:
          probe  layer_index  probe_score_mean  test_macro_f1_mean
linear_logistic           11          0.452301            0.285415
   mlp_1_hidden            7          0.455869            0.273473
   mlp_2_hidden            7          0.454193            0.279431
   mlp_3_hidden           11          0.446249            0.264002
[probe + 8982.02s] Output directory: /Volumes/Amirali/interEx/Qwen2-0.5B/goemo/Qwen-Qwen2-0.5B__goemo__linear_logistic+mlp_1_hidden+mlp_2_hidden+mlp_3_hidden__L25__R2__S5000__h5302b46bc7
[matrix] 3/18 | Qwen/Qwen2-0.5B | isear | e2dc842f
[probe +    0.00s] ================================================================================================
[probe +    0.00s] INITIALISING UNIFIED HIDDEN-STATE PROBE
[probe +    0.00s] ================================================================================================
[probe +    0.85s] Starting new trial: /Volumes/Amirali/interEx/Qwen2-0.5B/isear/Qwen-Qwen2-0.5B__isear__linear_logistic+mlp_1_hidden+mlp_2_hidden+mlp_3_hidden__L25__R2__S5000__he2dc842f4e
[probe +    1.22s] Model: Qwen/Qwen2-0.5B
[probe +    1.22s] Dataset artifact: isear
[probe +    1.22s] Hidden-state shape: (7532, 25, 896)
...
[probe +    1.22s] Selected layers: 25 | device: cpu
[probe +    1.22s] Alignment: text=verified | labels=unverified
[probe +    4.38s] No valid progress file found, starting fresh.
[probe +    4.38s] Resumed with 0 completed jobs.
Output is truncated. View as a scrollable element or open in a text editor. Adjust cell output settings...
Probing: 100%
 400/400 [40:09<00:00,  5.20s/fit]
[probe +    4.56s] ================================================================================================
[probe +    4.56s] PROBING EXPERIMENT
[probe +    4.56s] ================================================================================================
[probe +    4.56s] Question: how recoverable is the target from each frozen hidden-state layer?
[probe +    4.56s] repeats=2 | max_samples=5000 | layers=25 | probes=4
[probe + 2414.02s] All jobs completed. Generating final outputs.
[probe + 2425.87s] Complete run metadata saved: /Volumes/Amirali/interEx/Qwen2-0.5B/isear/Qwen-Qwen2-0.5B__isear__linear_logistic+mlp_1_hidden+mlp_2_hidden+mlp_3_hidden__L25__R2__S5000__he2dc842f4e/complete_run_metadata.json
[probe + 2425.89s] ================================================================================================
[probe + 2425.89s] FINAL RESULT
[probe + 2425.89s] ================================================================================================
[probe + 2425.90s] Final best layer table:
          probe  layer_index  probe_score_mean  test_macro_f1_mean
linear_logistic           11          0.571029            0.596981
   mlp_1_hidden            8          0.645029            0.664385
   mlp_2_hidden           16          0.639935            0.653573
   mlp_3_hidden           17          0.644681            0.667829
[probe + 2425.90s] Output directory: /Volumes/Amirali/interEx/Qwen2-0.5B/isear/Qwen-Qwen2-0.5B__isear__linear_logistic+mlp_1_hidden+mlp_2_hidden+mlp_3_hidden__L25__R2__S5000__he2dc842f4e
[matrix] 4/18 | Qwen/Qwen2-0.5B | tweet_eval_emotion | 647f98f9
[probe +    0.00s] ================================================================================================
[probe +    0.00s] INITIALISING UNIFIED HIDDEN-STATE PROBE
[probe +    0.00s] ================================================================================================
[probe +    0.29s] Starting new trial: /Volumes/Amirali/interEx/Qwen2-0.5B/tweet_eval_emotion/Qwen-Qwen2-0.5B__tweet_eval_emotion__linear_logistic+mlp_1_hidden+mlp_2_hidden+mlp_3_hidden__L25__R2__S5000__h647f98f9c3
[probe +    1.10s] Model: Qwen/Qwen2-0.5B
[probe +    1.10s] Dataset artifact: tweet_eval_emotion
[probe +    1.10s] Hidden-state shape: (5027, 25, 896)
...
[probe +    1.10s] Selected layers: 25 | device: cpu
[probe +    1.10s] Alignment: text=verified | labels=unverified
[probe +    1.38s] No valid progress file found, starting fresh.
[probe +    1.38s] Resumed with 0 completed jobs.
Output is truncated. View as a scrollable element or open in a text editor. Adjust cell output settings...
Probing: 100%
 400/400 [35:05<00:00,  6.16s/fit]
[probe +    1.46s] ================================================================================================
[probe +    1.46s] PROBING EXPERIMENT
[probe +    1.46s] ================================================================================================
[probe +    1.46s] Question: how recoverable is the target from each frozen hidden-state layer?
[probe +    1.46s] repeats=2 | max_samples=5000 | layers=25 | probes=4
[probe + 2107.13s] All jobs completed. Generating final outputs.
[probe + 2117.49s] Complete run metadata saved: /Volumes/Amirali/interEx/Qwen2-0.5B/tweet_eval_emotion/Qwen-Qwen2-0.5B__tweet_eval_emotion__linear_logistic+mlp_1_hidden+mlp_2_hidden+mlp_3_hidden__L25__R2__S5000__h647f98f9c3/complete_run_metadata.json
[probe + 2117.53s] ================================================================================================
[probe + 2117.53s] FINAL RESULT
[probe + 2117.53s] ================================================================================================
[probe + 2117.53s] Final best layer table:
          probe  layer_index  probe_score_mean  test_macro_f1_mean
linear_logistic           10          0.587895            0.632295
   mlp_1_hidden           11          0.668820            0.691046
   mlp_2_hidden            6          0.661823            0.684816
   mlp_3_hidden            6          0.661560            0.688493
[probe + 2117.55s] Output directory: /Volumes/Amirali/interEx/Qwen2-0.5B/tweet_eval_emotion/Qwen-Qwen2-0.5B__tweet_eval_emotion__linear_logistic+mlp_1_hidden+mlp_2_hidden+mlp_3_hidden__L25__R2__S5000__h647f98f9c3
[matrix] 5/18 | Qwen/Qwen2.5-0.5B | emotion | 0e3e9ef2
[probe +    0.00s] ================================================================================================
[probe +    0.00s] INITIALISING UNIFIED HIDDEN-STATE PROBE
[probe +    0.00s] ================================================================================================
[probe +    2.13s] Starting new trial: /Volumes/Amirali/interEx/Qwen2.5-0.5B/emotion/Qwen-Qwen2.5-0.5B__emotion__linear_logistic+mlp_1_hidden+mlp_2_hidden+mlp_3_hidden__L25__R2__S5000__h0e3e9ef2bb
[probe +    2.57s] Model: Qwen/Qwen2.5-0.5B
[probe +    2.57s] Dataset artifact: emotion
[probe +    2.57s] Hidden-state shape: (19999, 25, 896)
...
[probe +    2.57s] Selected layers: 25 | device: cpu
[probe +    2.57s] Alignment: text=verified | labels=unverified
[probe +    2.70s] No valid progress file found, starting fresh.
[probe +    2.70s] Resumed with 0 completed jobs.
Output is truncated. View as a scrollable element or open in a text editor. Adjust cell output settings...
Probing: 100%
 400/400 [58:55<00:00,  7.98s/fit]
[probe +    2.80s] ================================================================================================
[probe +    2.80s] PROBING EXPERIMENT
[probe +    2.80s] ================================================================================================
[probe +    2.80s] Question: how recoverable is the target from each frozen hidden-state layer?
[probe +    2.80s] repeats=2 | max_samples=5000 | layers=25 | probes=4
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
[probe + 3537.79s] All jobs completed. Generating final outputs.
[probe + 3549.58s] Complete run metadata saved: /Volumes/Amirali/interEx/Qwen2.5-0.5B/emotion/Qwen-Qwen2.5-0.5B__emotion__linear_logistic+mlp_1_hidden+mlp_2_hidden+mlp_3_hidden__L25__R2__S5000__h0e3e9ef2bb/complete_run_metadata.json
[probe + 3549.61s] ================================================================================================
[probe + 3549.61s] FINAL RESULT
[probe + 3549.61s] ================================================================================================
[probe + 3549.61s] Final best layer table:
          probe  layer_index  probe_score_mean  test_macro_f1_mean
linear_logistic            0          0.587389            0.600962
   mlp_1_hidden            1          0.643207            0.642781
   mlp_2_hidden            1          0.612510            0.637067
   mlp_3_hidden            1          0.631707            0.636231
[probe + 3549.64s] Output directory: /Volumes/Amirali/interEx/Qwen2.5-0.5B/emotion/Qwen-Qwen2.5-0.5B__emotion__linear_logistic+mlp_1_hidden+mlp_2_hidden+mlp_3_hidden__L25__R2__S5000__h0e3e9ef2bb
[matrix] 6/18 | Qwen/Qwen2.5-0.5B | goemo | e93157d3
[probe +    0.00s] ================================================================================================
[probe +    0.00s] INITIALISING UNIFIED HIDDEN-STATE PROBE
[probe +    0.00s] ================================================================================================
[probe +    5.78s] Starting new trial: /Volumes/Amirali/interEx/Qwen2.5-0.5B/goemo/Qwen-Qwen2.5-0.5B__goemo__linear_logistic+mlp_1_hidden+mlp_2_hidden+mlp_3_hidden__L25__R2__S5000__he93157d386
[probe +    6.66s] Model: Qwen/Qwen2.5-0.5B
[probe +    6.66s] Dataset artifact: goemo
[probe +    6.66s] Hidden-state shape: (54039, 25, 896)
[probe +    6.66s] Task type: multi_label | classes: 28
[probe +    6.66s] Selected layers: 25 | device: cpu
[probe +    6.66s] Alignment: text=verified | labels=unverified
[probe +    8.53s] No valid progress file found, starting fresh.
[probe +    8.53s] Resumed with 0 completed jobs.
Probing:  22%
 86/400 [28:56<1:24:18, 16.11s/fit]
[probe +    8.71s] ================================================================================================
[probe +    8.71s] PROBING EXPERIMENT
[probe +    8.71s] ================================================================================================
[probe +    8.71s] Question: how recoverable is the target from each frozen hidden-state layer?
[probe +    8.71s] repeats=2 | max_samples=5000 | layers=25 | probes=4